## Preprocessing of the "domy" dataset

In [2]:
# to access sibling directories
import sys
import os

import numpy as np
sys.path.append(os.path.abspath(".."))

from src.preprocessor_config import PreprocessorConfig
from src.preprocessor import Preprocessor
import pandas as pd
import yaml
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

path_input  = '../data/raw/domy.csv'
path_output = '../data/processed/'
path_config = '../config/preprocessor.yaml'

df = pd.read_csv(path_input)
with open(path_config, 'r') as f:
    cfg = PreprocessorConfig(**yaml.safe_load(f))

prepro = Preprocessor(df, cfg)
prepro.prepare_base_data()

# metryki
mae_scores = []
r2_scores = []

for X_train, X_test, y_train, y_test in prepro.get_folds():

    # trenuj
    model = LinearRegression()
    model.fit(X_train, y_train)

    # przewiduj
    preds = model.predict(X_test)

    # oceń
    mae_scores.append(mean_absolute_error(y_test, preds))
    r2_scores.append(r2_score(y_test, preds))

# Aggregate Results
print(f"Average MAE: ${np.mean(mae_scores):,.2f} (±{np.std(mae_scores):,.2f})")
print(f"Average R²: {np.mean(r2_scores):.4f}")

Average MAE: $16,247.47 (±543.74)
Average R²: 0.8586


### Steps
0. Drop IDs (not something the model should learn from)
1. Handling missing data (1 column MasVnrType with nulls 60%)
2. Handling outliers
3. Encoding categorical values, handling weird numerals parsed into strings
4. Splitting the dataset into train and test sets
5. Scaling the dataset